# GT-Loc Phase 1 — Colab Reproduction

This notebook implements **only Phase 1** of the study plan:

1. clone and freeze the official GT-Loc repository;
2. download and verify the pretrained checkpoint;
3. isolate the image and time encoders (no GPS retrieval branch);
4. encode the official `cvt.npy` time gallery (16,656 rows);
5. reproduce the SkyFinder time-prediction benchmark (TenCrop) when the images are supplied.

| Metric | Paper (GT-Loc, full model) | Reproduced earlier with this pipeline (progress deck) |
|---|---|---|
| ToY MAE (months) | 1.40 | 1.49 (1.4919) |
| ToD MAE (hours) | 2.72 | 2.67 (2.6704) |
| TPS (%) | 77.00 | 76.40 (76.4047) |

The cell after the evaluation prints the deltas from both reference sets. The deck values were produced in fp32. With `USE_FP16 = True` (the T4 default), small differences in the last decimals are expected. Set `USE_FP16 = False` to reproduce them exactly.

> **Runtime:** *Runtime → Change runtime type → T4 GPU*. The notebook expects CUDA. A CPU run works but is very slow.

> **Dataset note:** GT-Loc provides `metadata/skyfinder_val.csv` but does not redistribute SkyFinder images. The loader opens `SKYFINDER_ROOT / <csv path>`, where the CSV path looks like `204/20140817_164952.jpg`, so `SKYFINDER_ROOT` must be the folder whose children are camera-ID folders.

> **Outputs:** every run writes to a new folder `MyDrive/Colab Notebooks/CCTV/results/gtloc_phase1_<YYYYMMDD_HHMM>/` and never overwrites existing files.

In [ ]:
# Install only packages not normally included in a Colab runtime.
# Keep Colab's preinstalled torch/torchvision build.
%pip install -q "transformers>=4.45,<5" gdown pandas pillow tqdm matplotlib

import platform
import sys
print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())

In [ ]:
from pathlib import Path
import os
import subprocess

REPO_URL = "https://github.com/dshatwell23/gtloc.git"
# Repository HEAD inspected when this notebook was prepared (2026-09-29).
REPO_COMMIT = "6e7d9d2745d15a101b3b4dd05151807af77831ea"
WORKDIR = Path("/content") if Path("/content").exists() else Path.cwd()
REPO_DIR = WORKDIR / "gtloc"

if not (REPO_DIR / ".git").exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "origin", REPO_COMMIT], check=True)
subprocess.run(["git", "-C", str(REPO_DIR), "checkout", "--detach", REPO_COMMIT], check=True)

os.chdir(REPO_DIR)
print("GT-Loc repository:", REPO_DIR)
print("Pinned commit:", subprocess.check_output(
    ["git", "rev-parse", "HEAD"], text=True
).strip())

In [ ]:
import json
import random
import numpy as np
import pandas as pd
import torch
import transformers

SEED = 34
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
    RUNTIME = "cuda"
    torch.cuda.manual_seed_all(SEED)
    print("GPU:", torch.cuda.get_device_name(0))
else:
    DEVICE = torch.device("cpu")
    RUNTIME = "cpu"
    print("WARNING: no GPU. Select Runtime -> Change runtime type -> T4 GPU. CPU inference is very slow.")

# fp16 autocast for the CLIP image encoder on T4; retrieval and metrics stay in fp32.
USE_FP16 = RUNTIME == "cuda"

print(f"Runtime: {RUNTIME}")
print(f"Device: {DEVICE}")
print(f"fp16 autocast: {USE_FP16}")
print(f"torch: {torch.__version__}")
print(f"transformers: {transformers.__version__}")

In [ ]:
import hashlib
import gdown

CHECKPOINT_ID = "170wGyCYLSF3DWOvITIAbtN1OkmDlS4v6"
CKPT_DIR = REPO_DIR / "ckpts"
CKPT_PATH = CKPT_DIR / "gtloc.pt"
CKPT_DIR.mkdir(parents=True, exist_ok=True)

if not CKPT_PATH.exists() or CKPT_PATH.stat().st_size == 0:
    gdown.download(id=CHECKPOINT_ID, output=str(CKPT_PATH), quiet=False)

if not CKPT_PATH.exists() or CKPT_PATH.stat().st_size == 0:
    raise RuntimeError("Checkpoint download failed or produced an empty file.")

sha256 = hashlib.sha256()
with CKPT_PATH.open("rb") as stream:
    for block in iter(lambda: stream.read(8 * 1024 * 1024), b""):
        sha256.update(block)

print(f"Checkpoint: {CKPT_PATH}")
print(f"Size: {CKPT_PATH.stat().st_size / 2**30:.2f} GiB")
print(f"SHA-256: {sha256.hexdigest()}")

In [ ]:
import gc
import sys

sys.path.insert(0, str(REPO_DIR / "src"))
from model.image_encoder import ImageEncoder
from model.time_encoder import TimeEncoder

# Build only the two encoders needed for image-to-time retrieval. This avoids
# constructing or transferring either GPS gallery.
image_encoder = ImageEncoder(
    hidden_dim=768,
    embedding_dim=512,
    freeze_backbone=True,
)
time_encoder = TimeEncoder(
    sigma=[2**0, 2**4, 2**8],
    embedding_dim=512,
    dropout_prob=None,
)

try:
    raw_checkpoint = torch.load(CKPT_PATH, map_location="cpu", weights_only=True)
except Exception as exc:
    # PyTorch docs: weights_only=True is preferred. Official GT-Loc checkpoints
    # are trusted; fall back only if the file contains non-tensor globals.
    print(f"weights_only=True failed ({type(exc).__name__}); retrying with weights_only=False")
    raw_checkpoint = torch.load(CKPT_PATH, map_location="cpu", weights_only=False)
state_dict = raw_checkpoint.get("state_dict", raw_checkpoint) if isinstance(raw_checkpoint, dict) else raw_checkpoint
state_dict = {
    (key.removeprefix("module.")): value
    for key, value in state_dict.items()
}

image_state = {
    key.removeprefix("image_encoder."): value
    for key, value in state_dict.items()
    if key.startswith("image_encoder.")
}
time_state = {
    key.removeprefix("time_encoder."): value
    for key, value in state_dict.items()
    if key.startswith("time_encoder.")
}

if not image_state or not time_state:
    raise RuntimeError("Checkpoint does not contain the expected image_encoder/time_encoder keys.")

image_load = image_encoder.load_state_dict(image_state, strict=False)
time_load = time_encoder.load_state_dict(time_state, strict=False)

# Hugging Face versions can differ in non-parameter buffers such as position_ids.
# Unexpected trainable parameter mismatches are treated as fatal.
def trainable_mismatches(module, load_result, ignore_prefixes=()):
    trainable = {name for name, _ in module.named_parameters()}
    missing = sorted(
        name for name in (set(load_result.missing_keys) & trainable)
        if not any(name.startswith(prefix) for prefix in ignore_prefixes)
    )
    unexpected = sorted(set(load_result.unexpected_keys) & trainable)
    return missing, unexpected

img_missing, img_unexpected = trainable_mismatches(
    image_encoder, image_load, ignore_prefixes=("backbone.",)
)
time_missing, time_unexpected = trainable_mismatches(time_encoder, time_load)
if img_missing or img_unexpected or time_missing or time_unexpected:
    raise RuntimeError({
        "image_missing": img_missing,
        "image_unexpected": img_unexpected,
        "time_missing": time_missing,
        "time_unexpected": time_unexpected,
    })

backbone_missing = [
    key for key in image_load.missing_keys if key.startswith("backbone.")
]
image_encoder = image_encoder.eval().to(DEVICE)
time_encoder = time_encoder.eval().to(DEVICE)
del raw_checkpoint, state_dict, image_state, time_state
gc.collect()

print("Loaded isolated time-retrieval encoders.")
print(f"Expected omitted CLIP backbone keys: {len(backbone_missing)}")
print("Image trainable mismatches:", img_missing, img_unexpected)
print("Time trainable mismatches:", time_missing, time_unexpected)

In [ ]:
import torch.nn.functional as F

TIME_GALLERY_PATH = REPO_DIR / "src" / "model" / "time_galleries" / "cvt.npy"
time_gallery_cpu = torch.from_numpy(np.load(TIME_GALLERY_PATH)).float()

if time_gallery_cpu.ndim != 2 or time_gallery_cpu.shape[1] != 5:
    raise ValueError(f"Unexpected gallery shape: {tuple(time_gallery_cpu.shape)}")

# Released cvt.npy has 16,656 rows; encode in one shot (small MLP, fits easily on a T4).
with torch.inference_mode():
    time_gallery_features_cpu = F.normalize(
        time_encoder(time_gallery_cpu.to(DEVICE)), dim=1
    ).cpu()

print("Released time gallery shape:", tuple(time_gallery_cpu.shape))
print("Encoded gallery shape:", tuple(time_gallery_features_cpu.shape))
print(
    "Reproducibility note: the paper describes a 100k time gallery; "
    "the checked-in cvt.npy has", len(time_gallery_cpu), "rows."
)

## Supply the SkyFinder validation images

The official repository includes the validation CSV but not the images. Expected Drive layout:

```text
MyDrive/Colab Notebooks/CCTV/datasets/skyfinder/
├── 204/20140817_164952.jpg
├── 623/20130729_181156.jpg
└── 4795/20121102_130415.jpg
```

The loader joins `SKYFINDER_ROOT` with the CSV `path` (`<camera_id>/<file>.jpg`). If the images are one level deeper, under `skyfinder/amos/` as in the original AMOS/SkyFinder release, the next cell detects this and uses that folder. To use another location, change `SKYFINDER_ROOT`. The run stops with a missing-file report rather than evaluating a partial dataset.

In [ ]:
# Mount Drive only when running in Colab.
try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    print("Not running in Colab; using the configured local path.")

# >>> EDIT HERE if your SkyFinder images live elsewhere <<<
SKYFINDER_ROOT = Path("/content/drive/MyDrive/Colab Notebooks/CCTV/datasets/skyfinder")
SKYFINDER_CSV = REPO_DIR / "metadata" / "skyfinder_val.csv"

skyfinder_df = pd.read_csv(SKYFINDER_CSV)

# The loader opens SKYFINDER_ROOT / "<camera_id>/<file>.jpg". Accept an extra "amos/" level.
_probe = skyfinder_df["path"].iloc[0]
if not (SKYFINDER_ROOT / _probe).is_file() and (SKYFINDER_ROOT / "amos" / _probe).is_file():
    SKYFINDER_ROOT = SKYFINDER_ROOT / "amos"
    print("Using the 'amos' subfolder:", SKYFINDER_ROOT)
required_columns = {
    "path", "month", "day", "hour", "minute", "second"
}
missing_columns = required_columns - set(skyfinder_df.columns)
if missing_columns:
    raise ValueError(f"Metadata is missing columns: {sorted(missing_columns)}")

missing_paths = [
    relative_path
    for relative_path in skyfinder_df["path"]
    if not (SKYFINDER_ROOT / relative_path).is_file()
]
print(f"Metadata rows: {len(skyfinder_df):,}")
print(f"SkyFinder root: {SKYFINDER_ROOT}")
print(f"Missing images: {len(missing_paths):,}")
if missing_paths:
    raise FileNotFoundError(
        "SkyFinder validation set is incomplete. First missing paths: "
        + ", ".join(missing_paths[:10])
    )

In [ ]:
from torch.utils.data import DataLoader
from datasets.dataloader import GeoTemporalDataset, retvals
from datasets.transforms import get_transforms

# Official released evaluation uses TenCrop. 16 images x 10 crops fits a T4 (16 GB) in fp16;
# lower BATCH_SIZE to 8 on out-of-memory errors (or when USE_FP16 = False).
TENCROP = True
BATCH_SIZE = 16 if RUNTIME == "cuda" else 1
NUM_WORKERS = 2  # Colab standard runtimes have 2 vCPUs; overlaps Drive reads with GPU work.
GALLERY_CHUNK_SIZE = 8192

_, validation_transform = get_transforms(tencrop=TENCROP)
skyfinder_dataset = GeoTemporalDataset(
    metadata_path=str(SKYFINDER_CSV),
    imgs_path=str(SKYFINDER_ROOT),
    transform=validation_transform,
    fields=retvals.GEOTEMP,
    seed=SEED,
)
skyfinder_loader = DataLoader(
    skyfinder_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=(RUNTIME == "cuda"),
    persistent_workers=NUM_WORKERS > 0,
)

print(f"Images: {len(skyfinder_dataset):,}")
print(f"TenCrop: {TENCROP}")
print(f"Batch size: {BATCH_SIZE}")

In [ ]:
from tqdm.auto import tqdm

@torch.inference_mode()
def retrieve_time_indices(query_features, gallery_features, gallery_chunk_size=8192):
    """Exact cosine retrieval, averaging similarity over TenCrop views."""
    if query_features.ndim == 2:
        query_features = query_features[:, None, :]
    batch_size, crop_count, embedding_dim = query_features.shape
    flat_queries = query_features.reshape(batch_size * crop_count, embedding_dim)
    best_scores = torch.full((batch_size,), -torch.inf, device=query_features.device)
    best_indices = torch.zeros((batch_size,), dtype=torch.long, device=query_features.device)

    for start in range(0, len(gallery_features), gallery_chunk_size):
        gallery_chunk = gallery_features[start:start + gallery_chunk_size]
        similarities = flat_queries @ gallery_chunk.T
        similarities = similarities.reshape(batch_size, crop_count, -1).mean(dim=1)
        scores, local_indices = similarities.max(dim=1)
        improved = scores > best_scores
        best_scores = torch.where(improved, scores, best_scores)
        best_indices = torch.where(improved, start + local_indices, best_indices)

    return best_indices.cpu(), best_scores.cpu()


def circular_time_errors(reference_time, predicted_time):
    """Match the released GT-Loc evaluator exactly."""
    reference_month = reference_time[:, 0] + (reference_time[:, 1] - 1) / 31.0
    reference_hour = (
        reference_time[:, 2]
        + reference_time[:, 3] / 60.0
        + reference_time[:, 4] / 3600.0
    )
    predicted_month = predicted_time[:, 0] + (predicted_time[:, 1] - 1) / 31.0
    predicted_hour = (
        predicted_time[:, 2]
        + predicted_time[:, 3] / 60.0
        + predicted_time[:, 4] / 3600.0
    )
    month_error = torch.abs(reference_month - predicted_month)
    month_error = torch.minimum(month_error, 12 - month_error)
    hour_error = torch.abs(reference_hour - predicted_hour)
    hour_error = torch.minimum(hour_error, 24 - hour_error)
    return month_error, hour_error


@torch.inference_mode()
def evaluate_skyfinder(loader):
    gallery_features = time_gallery_features_cpu.to(DEVICE)
    records = []
    row_offset = 0

    for images, metadata in tqdm(loader, desc="SkyFinder"):
        images = images.to(DEVICE, non_blocking=True)
        if TENCROP:
            batch_size, crop_count, channels, height, width = images.shape
            flat_images = images.reshape(-1, channels, height, width)
        else:
            batch_size = images.shape[0]
            crop_count = 1
            flat_images = images

        # fp16 autocast only for the CLIP image encoder; normalise and retrieve in fp32.
        with torch.autocast("cuda", dtype=torch.float16, enabled=USE_FP16):
            image_features = image_encoder(flat_images)
        image_features = F.normalize(image_features.float(), dim=1)
        image_features = image_features.reshape(batch_size, crop_count, -1)
        predicted_indices, similarities = retrieve_time_indices(
            image_features,
            gallery_features,
            gallery_chunk_size=GALLERY_CHUNK_SIZE,
        )
        predicted_time = time_gallery_cpu[predicted_indices]
        reference_time = metadata["time"].cpu()
        month_error, hour_error = circular_time_errors(reference_time, predicted_time)

        paths = list(metadata["path"])
        for index in range(batch_size):
            records.append({
                "path": paths[index],
                "true_month": int(reference_time[index, 0]),
                "true_day": int(reference_time[index, 1]),
                "true_hour": int(reference_time[index, 2]),
                "true_minute": int(reference_time[index, 3]),
                "true_second": int(reference_time[index, 4]),
                "pred_month": int(predicted_time[index, 0]),
                "pred_day": int(predicted_time[index, 1]),
                "pred_hour": int(predicted_time[index, 2]),
                "pred_minute": int(predicted_time[index, 3]),
                "pred_second": int(predicted_time[index, 4]),
                "month_error": float(month_error[index]),
                "hour_error": float(hour_error[index]),
                "cosine_similarity": float(similarities[index]),
            })
        row_offset += batch_size

    return pd.DataFrame.from_records(records)

print("Evaluation functions ready.")

In [ ]:
predictions = evaluate_skyfinder(skyfinder_loader)

mean_month_error = predictions["month_error"].mean()
mean_hour_error = predictions["hour_error"].mean()
tps = 100.0 * (
    1.0 - np.sqrt(
        ((mean_month_error / 6.0) ** 2 + (mean_hour_error / 12.0) ** 2) / 2.0
    )
)

REFERENCE = {  # GT-Loc paper (full model)
    "mean_month_error": 1.40,
    "mean_hour_error": 2.72,
    "tps_percent": 77.00,
}
REPRODUCED_DECK = {  # earlier fp32 run of this pipeline (progress deck / report)
    "mean_month_error": 1.4919,
    "mean_hour_error": 2.6704,
    "tps_percent": 76.4047,
}
results = {
    "n_images": int(len(predictions)),
    "mean_month_error": float(mean_month_error),
    "mean_hour_error": float(mean_hour_error),
    "tps_percent": float(tps),
    "reference": REFERENCE,
    "delta_from_reference": {
        "mean_month_error": float(mean_month_error - REFERENCE["mean_month_error"]),
        "mean_hour_error": float(mean_hour_error - REFERENCE["mean_hour_error"]),
        "tps_percent": float(tps - REFERENCE["tps_percent"]),
    },
    "reproduced_deck": REPRODUCED_DECK,
    "delta_from_reproduced_deck": {
        key: float(value - REPRODUCED_DECK[key])
        for key, value in {
            "mean_month_error": mean_month_error,
            "mean_hour_error": mean_hour_error,
            "tps_percent": tps,
        }.items()
    },
    "runtime": RUNTIME,
    "device": str(DEVICE),
    "tencrop": TENCROP,
    "batch_size": BATCH_SIZE,
    "fp16_autocast": USE_FP16,
    "seed": SEED,
    "repository_commit": REPO_COMMIT,
    "checkpoint_sha256": sha256.hexdigest(),
    "time_gallery_rows": int(len(time_gallery_cpu)),
    "torch_version": torch.__version__,
    "transformers_version": transformers.__version__,
}

print(json.dumps(results, indent=2))

In [ ]:
# Persist all per-image predictions and provenance in a NEW timestamped folder (never overwrite).
from datetime import datetime

RUN_STAMP = datetime.now().strftime("%Y%m%d_%H%M")
RESULTS_ROOT = Path("/content/drive/MyDrive/Colab Notebooks/CCTV/results")
if not Path("/content/drive/MyDrive").exists():
    RESULTS_ROOT = WORKDIR / "results"
OUTPUT_DIR = RESULTS_ROOT / f"gtloc_phase1_{RUN_STAMP}"
RESULTS_ROOT.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(exist_ok=False)  # fails rather than reusing an existing run folder

predictions_path = OUTPUT_DIR / "skyfinder_predictions.csv"
results_path = OUTPUT_DIR / "skyfinder_reproduction_results.json"
provenance_path = OUTPUT_DIR / "environment.txt"
for _path in (predictions_path, results_path, provenance_path):
    if _path.exists():
        raise FileExistsError(f"Refusing to overwrite {_path}")

predictions.to_csv(predictions_path, index=False)
results_path.write_text(json.dumps(results, indent=2) + "\n", encoding="utf-8")

try:
    frozen_packages = subprocess.check_output(
        [sys.executable, "-m", "pip", "freeze"], text=True
    )
except subprocess.CalledProcessError:
    frozen_packages = "pip freeze failed\n"
provenance_path.write_text(frozen_packages, encoding="utf-8")

print("Saved:")
print(" -", predictions_path)
print(" -", results_path)
print(" -", provenance_path)